# Round 5 EDA — Cherry Picking Winners

Goal: figure out what predicts intraday drift for each of the 10 product groups, before committing to any model.

Plan:
1. Load & sanity check
2. Build generic features
3. Per-group visual EDA
4. Within-group correlation & lead-lag
5. Imbalance → drift relationship
6. Group-specific structure (PANEL areas, PEBBLES ladder, SNACKPACK persistence)
7. Feature → future-return correlation table
8. Linear baseline

If anything in 5–7 shows a clean signal, that may already be your strategy.

## 1. Setup & load

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

DATA_DIR = '.'  # adjust if needed
DAYS = [2, 3, 4]

In [ ]:
def load_prices():
    out = []
    for d in DAYS:
        df = pd.read_csv(f'{DATA_DIR}/prices_round_5_day_{d}.csv', sep=';')
        df['day'] = d
        out.append(df)
    return pd.concat(out, ignore_index=True)

def group_of(prod):
    for k in ['GALAXY','SLEEP_POD','MICROCHIP','PEBBLES','ROBOT',
             'UV_VISOR','TRANSLATOR','PANEL','OXYGEN_SHAKE','SNACKPACK']:
        if prod.startswith(k): return k

prices = load_prices()
prices['group'] = prices['product'].map(group_of)
prices = prices.sort_values(['day','product','timestamp']).reset_index(drop=True)
prices.shape, prices['product'].nunique(), prices['group'].nunique()

## 2. Sanity check

In [ ]:
# missing levels, timestamp regularity, unique products per group
print('rows per (day, product):')
print(prices.groupby(['day','product']).size().describe())
print('\nproducts per group:')
print(prices.groupby('group')['product'].nunique())
print('\nL2 / L3 missingness:')
for c in ['bid_price_2','bid_price_3','ask_price_2','ask_price_3']:
    print(f'  {c}: {prices[c].isna().mean():.2%}')

## 3. Generic features

Built per (day, product). Group-relative versions added after.

In [ ]:
def add_features(df):
    df = df.copy()
    bv1, av1 = df['bid_volume_1'].fillna(0), df['ask_volume_1'].fillna(0)
    b1, a1 = df['bid_price_1'], df['ask_price_1']
    
    df['mid'] = (b1 + a1) / 2
    df['micro'] = (b1 * av1 + a1 * bv1) / (bv1 + av1).replace(0, np.nan)
    df['micro_dev'] = df['micro'] - df['mid']
    df['spread'] = a1 - b1
    df['imb1'] = (bv1 - av1) / (bv1 + av1).replace(0, np.nan)
    
    # L2/L3 totals (treat missing as 0)
    bv = df[['bid_volume_1','bid_volume_2','bid_volume_3']].fillna(0).sum(axis=1)
    av = df[['ask_volume_1','ask_volume_2','ask_volume_3']].fillna(0).sum(axis=1)
    df['imb_total'] = (bv - av) / (bv + av).replace(0, np.nan)
    df['depth_total'] = bv + av
    
    # log return at lags (per product, per day)
    df['logmid'] = np.log(df['mid'])
    for lag in [1, 5, 20, 100, 500]:
        df[f'ret_{lag}'] = df.groupby(['day','product'])['logmid'].diff(lag)
    
    # rolling vol (per product per day)
    df['rv_500'] = (df.groupby(['day','product'])['ret_1']
                      .rolling(500).std().reset_index(level=[0,1], drop=True))
    return df

feat = add_features(prices)
feat[['product','timestamp','mid','micro_dev','imb1','imb_total','rv_500']].head()

In [ ]:
# group-relative versions: x − mean of the group's 5 products at the same timestamp
def add_group_relative(df, cols):
    grp_mean = df.groupby(['day','timestamp','group'])[cols].transform('mean')
    for c in cols:
        df[f'{c}_gr'] = df[c] - grp_mean[c]
    return df

feat = add_group_relative(feat, ['imb1','imb_total','micro_dev','spread','rv_500'])

## 4. Per-group mid plots

Eyeball: do members diverge gradually, jump, or drift smoothly? Is the winner obvious early?

In [ ]:
def plot_group(df, group, day):
    sub = df[(df['group']==group) & (df['day']==day)]
    fig, ax = plt.subplots(figsize=(11, 4))
    for prod, s in sub.groupby('product'):
        ax.plot(s['timestamp'], s['mid'], label=prod.replace(group+'_',''), lw=0.8)
    ax.set_title(f'{group} — day {day}')
    ax.legend(fontsize=8, loc='best')
    plt.tight_layout(); plt.show()

# loop all groups, all days — adjust to taste
for g in feat['group'].unique():
    for d in DAYS:
        plot_group(feat, g, d)

## 5. Within-group return correlations & lead-lag

Look for products that lead others inside the same group.

In [ ]:
def group_return_corr(df, group, day, lag=0):
    sub = df[(df['group']==group) & (df['day']==day)]
    wide = sub.pivot(index='timestamp', columns='product', values='ret_5')
    if lag != 0:
        # correlate col_i(t) with col_j(t+lag); positive lag = j leads i
        wide_shifted = wide.shift(-lag)
        return wide.corrwith(wide_shifted, axis=0)  # element-wise won't help; use full matrix below
    return wide.corr()

# contemporaneous correlation matrix per group, day 2
for g in feat['group'].unique():
    print(f'\n=== {g} (day 2) ===')
    print(group_return_corr(feat, g, 2).round(2))

In [ ]:
# lead-lag: for each pair (i, j) in a group, find lag k that maximises corr(ret_i(t), ret_j(t+k))
def lead_lag(df, group, day, lags=range(-50, 51, 5)):
    sub = df[(df['group']==group) & (df['day']==day)]
    wide = sub.pivot(index='timestamp', columns='product', values='ret_5').dropna()
    prods = wide.columns.tolist()
    best = {}
    for i in prods:
        for j in prods:
            if i >= j: continue
            corrs = [wide[i].corr(wide[j].shift(-k)) for k in lags]
            k_best = lags[int(np.argmax(np.abs(corrs)))]
            best[(i,j)] = (k_best, corrs[int(np.argmax(np.abs(corrs)))])
    return best

# example: just one group to keep output short
ll = lead_lag(feat, 'MICROCHIP', 2)
for k, v in ll.items():
    print(f'{k[0]:25s} vs {k[1]:25s}  best lag={v[0]:+d}  corr={v[1]:+.2f}')

## 6. Imbalance → drift sign

Core question: does early-day book imbalance predict end-of-day drift sign?

In [ ]:
def early_signal_vs_drift(df, signal_col, warmup_ticks=2000):
    rows = []
    for (day, prod), s in df.groupby(['day','product']):
        s = s.sort_values('timestamp')
        warmup = s.iloc[:warmup_ticks]
        end_drift = s['mid'].iloc[-1] - s['mid'].iloc[0]
        rows.append({'day': day, 'product': prod, 'group': group_of(prod),
                     'signal': warmup[signal_col].mean(),
                     'drift': end_drift})
    r = pd.DataFrame(rows)
    print(f'\n--- {signal_col} (first {warmup_ticks} ticks) vs end-of-day drift ---')
    print('overall pearson:', pearsonr(r['signal'].fillna(0), r['drift'])[0].round(3))
    print('overall spearman:', spearmanr(r['signal'].fillna(0), r['drift'])[0].round(3))
    print('\nby group (spearman):')
    for g, sub in r.groupby('group'):
        rho, _ = spearmanr(sub['signal'].fillna(0), sub['drift'])
        print(f'  {g:15s}  rho={rho:+.2f}  n={len(sub)}')
    return r

for sig in ['imb1', 'imb_total', 'micro_dev', 'imb1_gr', 'micro_dev_gr']:
    early_signal_vs_drift(feat, sig, warmup_ticks=2000)

In [ ]:
# also try different warmup windows — when does the signal stabilise?
for w in [200, 500, 1000, 2000, 5000]:
    r = early_signal_vs_drift(feat, 'imb1_gr', warmup_ticks=w)

## 7. Group-specific structure

### 7a. PANEL area arbitrage

PANEL_1×2, 1×4, 2×2, 2×4, 4×4 → areas 2, 4, 4, 8, 16. If priced as a multiple of a base, residuals from a linear fit are tradable.

In [ ]:
PANEL_AREA = {'PANEL_1X2': 2, 'PANEL_1X4': 4, 'PANEL_2X2': 4,
              'PANEL_2X4': 8, 'PANEL_4X4': 16}

panels = feat[feat['group']=='PANEL'].copy()
panels['area'] = panels['product'].map(PANEL_AREA)
# at each timestamp, regress mid on area, look at residuals
snap = panels.pivot_table(index=['day','timestamp'], columns='product', values='mid')
areas = pd.Series(PANEL_AREA)
# implied unit price per timestamp = mid / area; check stability
unit = snap.div(areas, axis=1)
print('std of unit price across panels per timestamp (mean over time):')
print(unit.std(axis=1).mean(), 'vs typical mid level ~10000')
unit.std(axis=1).rolling(500).mean().plot(figsize=(11,3), title='dispersion of mid/area across PANELs'); plt.show()

### 7b. PEBBLES size ladder

XS, S, M, L, XL is ordered. Check if ordering matters — e.g., does mid track size, or is it just a label?

In [ ]:
PEB_ORDER = {'PEBBLES_XS': 0, 'PEBBLES_S': 1, 'PEBBLES_M': 2,
             'PEBBLES_L': 3, 'PEBBLES_XL': 4}
peb = feat[feat['group']=='PEBBLES'].copy()
peb['size_idx'] = peb['product'].map(PEB_ORDER)
# does end-of-day drift correlate with size index?
for d in DAYS:
    sub = peb[peb['day']==d]
    drift = sub.groupby(['product','size_idx'])['mid'].agg(lambda x: x.iloc[-1]-x.iloc[0]).reset_index()
    rho, _ = spearmanr(drift['size_idx'], drift['mid'])
    print(f'day {d}: spearman(size_idx, drift) = {rho:+.2f}')
    print(drift.to_string(index=False))

### 7c. SNACKPACK persistence

It was the only group with positive day-pair rank correlations. Check if that means it's stationary, or if the *winners* just stay winners.

In [ ]:
snack = feat[feat['group']=='SNACKPACK']
for d in DAYS:
    sub = snack[snack['day']==d]
    print(f'\nday {d} drift per product:')
    print(sub.groupby('product')['mid'].agg(lambda x: x.iloc[-1]-x.iloc[0]).sort_values())
    print(f'  per-tick ret std (median across products): {sub.groupby("product")["ret_1"].std().median():.6f}')
    print(f'  PEBBLES same metric for ref: {feat[(feat["group"]=="PEBBLES") & (feat["day"]==d)].groupby("product")["ret_1"].std().median():.6f}')

## 8. Feature → future-return correlation table

Quantitative ranking of which features predict short-horizon returns, per group.

In [ ]:
# add forward returns
for h in [100, 500, 2000]:
    feat[f'fwd_{h}'] = (feat.groupby(['day','product'])['logmid']
                          .shift(-h) - feat['logmid'])

FEATURES = ['imb1','imb_total','micro_dev','spread','rv_500',
            'ret_5','ret_20','ret_100',
            'imb1_gr','imb_total_gr','micro_dev_gr','spread_gr','rv_500_gr']

def feat_target_corr(df, horizon, by_group=True):
    rows = []
    target = f'fwd_{horizon}'
    iter_by = df.groupby('group') if by_group else [('ALL', df)]
    for g, sub in iter_by:
        for f in FEATURES:
            s = sub[[f, target]].dropna()
            if len(s) < 1000: continue
            rho, _ = spearmanr(s[f], s[target])
            rows.append({'group': g, 'feature': f, 'horizon': horizon, 'spearman': rho})
    return pd.DataFrame(rows)

tbl = pd.concat([feat_target_corr(feat, h) for h in [100, 500, 2000]])
# pivot to view: rows = feature, cols = (group, horizon)
pivot = tbl.pivot_table(index='feature', columns=['group','horizon'], values='spearman')
pivot.style.background_gradient(cmap='RdBu_r', vmin=-0.1, vmax=0.1, axis=None)

In [ ]:
# overall (no grouping)
tbl_all = pd.concat([feat_target_corr(feat, h, by_group=False) for h in [100, 500, 2000]])
tbl_all.pivot(index='feature', columns='horizon', values='spearman').round(3)

## 9. Linear baseline

If a tiny linear model on the top-3 features already produces decent IC, you don't need a NN.

In [ ]:
from sklearn.linear_model import Ridge

HORIZON = 500
TARGET = f'fwd_{HORIZON}'

data = feat.dropna(subset=FEATURES + [TARGET]).copy()
# train on day 2+3 first 80%, val on last 20% of those days, test = day 4
def split(df):
    train, val, test = [], [], []
    for (d, p), s in df.groupby(['day','product']):
        s = s.sort_values('timestamp')
        if d == 4:
            test.append(s)
        else:
            cut = int(len(s) * 0.8)
            train.append(s.iloc[:cut])
            val.append(s.iloc[cut:])
    return pd.concat(train), pd.concat(val), pd.concat(test)

tr, va, te = split(data)
model = Ridge(alpha=1.0).fit(tr[FEATURES], tr[TARGET])

for name, s in [('train', tr), ('val', va), ('test=day4', te)]:
    pred = model.predict(s[FEATURES])
    rho, _ = spearmanr(pred, s[TARGET])
    hit = ((pred > 0) == (s[TARGET] > 0)).mean()
    print(f'{name:12s}  spearman={rho:+.3f}  hit-rate={hit:.3f}  n={len(s)}')

print('\ncoefficients:')
print(pd.Series(model.coef_, index=FEATURES).sort_values(key=abs, ascending=False).round(4))

## What to do with the output

- If section 6 shows clean per-group spearman > 0.3 between an early-day signal and end-day drift → rule-based strategy may suffice.
- If section 8 heatmap has features lighting up consistently for the same groups → those are your inputs.
- If section 9 ridge has hit-rate > 0.52 on day 4 → you have a baseline. NN target: beat it on the same val/test split.
- If everything is near zero → re-examine; signal may live at different horizons or in non-linear interactions.